<a href="https://colab.research.google.com/github/SwethaPalakonda/Conversational-Image-Classifier/blob/main/ARNVA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers openai Pillow

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from IPython.display import display
import openai
import glob
import os

In [ ]:
labels = ["healthy food", "junk food", "fruit", "snack", "vegetable", "dessert"]

In [ ]:
def classify_image(image: Image.Image):
    inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)
    outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1).detach().numpy()[0]
    best_idx = np.argmax(probs)
    return labels[best_idx], probs[best_idx]

In [ ]:
import requests

def generate_response(classification, question):
    api_key = "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"  # Your OpenRouter API key
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://your-project-or-email",  # Optional but recommended (you can use your email)
    }

    prompt = (
        f"This is a {classification}.\n\n"
        f"Question: {question}\n"
        f"Respond as a friendly food expert with reasoning and healthy tips."
    )

    data = {
        "model": "mistralai/mistral-7b-instruct",  # Free and fast
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(url, headers=headers, json=data)

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print("Error:", response.status_code, response.json())
        return "Sorry, I couldn't process that."




In [ ]:
!wget -nc http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
!tar -xzf food-101.tar.gz

--2025-04-13 15:49:01--  http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://data.vision.ee.ethz.ch/cvl/food-101.tar.gz [following]
--2025-04-13 15:49:01--  https://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4996278331 (4.7G) [application/x-gzip]
Saving to: ‘food-101.tar.gz’

food-101.tar.gz     100%[===================>]   4.65G  26.4MB/s    in 3m 2s   

2025-04-13 15:52:04 (26.1 MB/s) - ‘food-101.tar.gz’ saved [4996278331/4996278331]



In [ ]:
image_paths = glob.glob("food-101/images/**/*.jpg", recursive=True)[:5]

In [ ]:
#from transformers import CLIPProcessor, CLIPModel
#from PIL import Image

# Make sure to replace 'generate_response' function here as shown above

# Example of the main loop that processes images
#for path in image_paths:
    #print(f"\n Image: {path}")
    #image = Image.open(path).convert("RGB")  # Open image
    #display(image)  # Display image

    # Your image classification logic
    #label, confidence = classify_image(image)
    #print(f"\n Prediction: {label} ({confidence*100:.2f}% confidence)")

    # Ask user a question about the food
    #question = input("\n Ask a question about this food: ")
    #if question:
        # Call the new OpenRouter-based function for answering questions
        #answer = generate_response(label, question)
        #print("\n GPT says:\n", answer)



In [ ]:
!pip install streamlit transformers torch Pillow openai

In [ ]:
code = '''
# Your full streamlit-based code goes here
# Example:
import streamlit as st
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
import requests

st.title("🍎 Food Image Classifier + Expert Advice")

labels = ["healthy food", "junk food", "fruit", "snack", "vegetable", "dessert"]
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def classify_image(image):
    inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)
    outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1).detach().numpy()[0]
    best_idx = np.argmax(probs)
    return labels[best_idx], probs[best_idx]

def generate_response(classification, question):
    api_key = "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"  # Your OpenRouter key here
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://your-project-or-email",
    }

    prompt = f"""This is a {classification}.
    Question: {question}
    Respond as a friendly food expert with reasoning and healthy tips."""


    data = {
        "model": "mistralai/mistral-7b-instruct",
        "messages": [{"role": "user", "content": prompt}]
    }

    response = requests.post(url, headers=headers, json=data)
    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        return "Sorry, I couldn't get a response."

uploaded_file = st.file_uploader("Upload a food image", type=["jpg", "png", "jpeg"])
if uploaded_file:
    image = Image.open(uploaded_file).convert("RGB")
    st.image(image, caption="Uploaded Image", use_column_width=True)

    label, confidence = classify_image(image)
    st.markdown(f"### 🍽️ Prediction: `{label}` ({confidence*100:.2f}% confidence)")

    question = st.text_input("Ask a question about this food:")
    if question:
        answer = generate_response(label, question)
        st.markdown("### 🤖 GPT says:")
        st.markdown(answer)
'''
with open("app.py", "w") as f:
    f.write(code)


In [ ]:
from pyngrok import conf
conf.get_default().auth_token = "2vd2rjnPTGIGQvH49N0izwAHZmR_4mRgJhbHLEs6i9kRs5YdP"


In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Open new tunnel
public_url = ngrok.connect(8501)
print("🔗 Streamlit App URL:", public_url)

# Start Streamlit
!streamlit run app.py &> /dev/null &


🔗 Streamlit App URL: NgrokTunnel: "https://5732-34-46-41-2.ngrok-free.app" -> "http://localhost:8501"
